In [5]:
import pandas as pd
import numpy as np
import ssl

# Bypass SSL Certificate Error
ssl._create_default_https_context = ssl._create_unverified_context

team_id = "WSH"
url = f"https://www.hockey-reference.com/teams/{team_id}/history.html"

# Scrape the table
tables = pd.read_html(url)
df_raw = tables[0]

# Display the table
print(f"Table Shape: {df_raw.shape}")
df_raw.head()

Table Shape: (51, 17)


,Season,Lg,Team,GP,W,L,T,OL,PTS,PTS%,SRS,SOS,Finish,Playoffs,Coaches,Division,Conference
0,2025-26,NHL,Washington Capitals,82,43,30,NaN,9.0,95,0.579,0.25,0.02,4th of 8,NaN,S. Carbery (43-30-9),Metropolitan,Eastern
1,2024-25,NHL,Washington Capitals*,82,51,22,NaN,9.0,111,0.677,0.66,-0.03,1st of 8,Lost NHL Second Round,S. Carbery (51-22-9),Metropolitan,Eastern
2,2023-24,NHL,Washington Capitals*,82,40,31,NaN,11.0,91,0.555,-0.41,0.04,4th of 8,Lost NHL First Round,S. Carbery (40-31-11),Metropolitan,Eastern
3,2022-23,NHL,Washington Capitals,82,35,37,NaN,10.0,80,0.488,-0.10,0.02,6th of 8,NaN,P. Laviolette (35-37-10),Metropolitan,Eastern
4,2021-22,NHL,Washington Capitals*,82,44,26,NaN,12.0,100,0.610,0.35,-0.02,4th of 8,Lost NHL First Round,P. Laviolette (44-26-12),Metropolitan,Eastern


In [6]:
col_map = {
    df_raw.columns[0]: 'Season',
    df_raw.columns[3]: 'GP',
    df_raw.columns[4]: 'W',
    df_raw.columns[5]: 'L',
    df_raw.columns[13]: 'Playoffs'
}

df_raw = df_raw.rename(columns=col_map)

# Clean and Filter
# Keep only the rows that look like '2016-17' (removes mid-table headers)
df_raw = df_raw[df_raw['Season'].str.contains('-', na=False)].copy()

# Extract Season Start Year
df_raw['Season_Start'] = df_raw['Season'].str[:4].astype(int)

# Filter for the 2005-2017 window
study_window = df_raw[(df_raw['Season_Start'] >= 2005) & (df_raw['Season_Start'] <= 2024)].copy()

# TEST: Check the counts and look at the whole window
print(f"Seasons found: {len(study_window)}")

# Display the whole window to verify
study_window[['Season', 'GP', 'W', 'L', 'Playoffs']]

Seasons found: 20


,Season,GP,W,L,Playoffs
1,2024-25,82,51,22,Lost NHL Second Round
2,2023-24,82,40,31,Lost NHL First Round
3,2022-23,82,35,37,NaN
4,2021-22,82,44,26,Lost NHL First Round
5,2020-21,56,36,15,Lost NHL First Round
6,2019-20,69,41,20,Lost NHL First Round
7,2018-19,82,48,26,Lost NHL First Round
8,2017-18,82,49,26,Won Stanley Cup Final
9,2016-17,82,55,19,Lost NHL Second Round
10,2015-16,82,56,18,Lost NHL Second Round


In [7]:
study_window["Abbr"] = "WSH"

# Rename to match your main dataset exactly
study_window = study_window.rename(columns={
    "Playoffs": "Playoff_Result"
})

# Keep only the correct columns + order
wsh_df = study_window[["Abbr", "Season", "GP", "W", "Playoff_Result"]]

print(wsh_df.head())

  Abbr   Season  GP   W         Playoff_Result
1  WSH  2024-25  82  51  Lost NHL Second Round
2  WSH  2023-24  82  40   Lost NHL First Round
3  WSH  2022-23  82  35                    NaN
4  WSH  2021-22  82  44   Lost NHL First Round
5  WSH  2020-21  56  36   Lost NHL First Round


In [8]:
csv_path = "../data/nhl_performance.csv"

wsh_df.to_csv(csv_path, mode="a", header=False, index=False)

print("💾 Washington successfully appended")

💾 Washington successfully appended


In [9]:
df = pd.read_csv("../data/nhl_performance.csv")

print(df[df["Abbr"] == "WSH"].tail())
print(f"Total WSH rows: {len(df[df['Abbr'] == 'WSH'])}")

    Abbr   Season  GP   W                      Playoff_Result
594  WSH  2009-10  82  54  Lost NHL Conference Quarter-Finals
595  WSH  2008-09  82  50     Lost NHL Conference Semi-Finals
596  WSH  2007-08  82  43  Lost NHL Conference Quarter-Finals
597  WSH  2006-07  82  28                                 NaN
598  WSH  2005-06  82  29                                 NaN
Total WSH rows: 20


In [13]:
# 1. Convert stats to numeric (handles the 2012-13 season correctly)
study_window['W'] = pd.to_numeric(study_window['W'], errors='coerce')
study_window['GP'] = pd.to_numeric(study_window['GP'], errors='coerce')

# 2. Basic Win Stats
total_wins = study_window['W'].sum()
total_games = study_window['GP'].sum()
win_pct = total_wins / total_games

# 3. Playoff Progress Map
progress_map = {
    'Won Stanley Cup': 5,
    'Lost Stanley Cup Final': 4,
    'Lost NHL Finals': 4,
    'Lost NHL Conference Finals': 3,
    'Lost NHL Second Round': 2,
    'Lost NHL Conference Semi-Finals': 2,
    'Lost NHL First Round': 1,
    'Lost NHL Conference Quarter-Finals': 1
}

# Apply the map
study_window['Progress_Level'] = study_window['Playoffs'].map(progress_map).fillna(0)

# 4. Verification Check
print(f"Analysis for {team_id}:")
print(f"Total Window GP: {total_games}")
print(f"Total Window Wins: {total_wins}")
print(f"Window Win %: {win_pct:.4f}")
print(f"Total Progress Points: {study_window['Progress_Level'].sum()}")
print(f"Playoff Appearances: {len(study_window[study_window['Progress_Level'] > 0])}")

# Let's see the breakdown to make sure no points were dropped
study_window[['Season', 'Playoffs', 'Progress_Level']]

Analysis for NYR:
Total Window GP: 1032
Total Window Wins: 556
Window Win %: 0.5388
Total Progress Points: 22.0
Playoff Appearances: 11


,Season,Playoffs,Progress_Level
8,2017-18,NaN,0.0
9,2016-17,Lost NHL Second Round,2.0
10,2015-16,Lost NHL First Round,1.0
11,2014-15,Lost NHL Conference Finals,3.0
12,2013-14,Lost Stanley Cup Final,4.0
13,2012-13,Lost NHL Conference Semi-Finals,2.0
14,2011-12,Lost NHL Conference Finals,3.0
15,2010-11,Lost NHL Conference Quarter-Finals,1.0
16,2009-10,NaN,0.0
17,2008-09,Lost NHL Conference Quarter-Finals,1.0


In [20]:
import pandas as pd
import time
import ssl

# 1. Setup
ssl._create_default_https_context = ssl._create_unverified_context

teams = [
    'ANA', 'ARI', 'BOS', 'BUF', 'CGY', 'CAR', 'CHI', 'COL', 'CBJ', 'DAL', 
    'DET', 'EDM', 'FLA', 'LAK', 'MIN', 'MTL', 'NSH', 'NJD', 'NYI', 'NYR', 
    'OTT', 'PHI', 'PIT', 'SJS', 'STL', 'TBL', 'TOR', 'VAN', 'WPG', 'WSH'
]

custom_url_ids = {'ARI': 'PHX'}

# QUADRATIC WEIGHTING: Removes the 'finickiness' with a mathematical principle (n^2)
principled_weight_map = {
    'Won Stanley Cup Final': 25,                 
    'Won Stanley Cup': 25,                 
    'Lost Stanley Cup Final': 16,          
    'Lost NHL Finals': 16,                 
    'Lost NHL Conference Finals': 9,       
    'Lost NHL Second Round': 4,            
    'Lost NHL Conference Semi-Finals': 4,  
    'Lost NHL First Round': 1,             
    'Lost NHL Conference Quarter-Finals': 1 
}

performance_records = []

# 2. Execution
for team_abbr in teams:
    url_id = custom_url_ids.get(team_abbr, team_abbr)
    url = f"https://www.hockey-reference.com/teams/{url_id}/history.html"
    
    try:
        try:
            tables = pd.read_html(url)
        except:
            url = f"https://www.hockey-reference.com/teams/{url_id}/"
            tables = pd.read_html(url)
            
        df = tables[0]
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(-1)
            
        col_map = {df.columns[0]: 'Season', df.columns[3]: 'GP', df.columns[4]: 'W', df.columns[13]: 'Playoffs'}
        df = df.rename(columns=col_map)
        
        df = df[df['Season'].str.contains('-', na=False)].copy()
        df['Season_Start'] = df['Season'].str[:4].astype(int)
        window = df[(df['Season_Start'] >= 2005) & (df['Season_Start'] <= 2017)].copy()
        
        window['W'] = pd.to_numeric(window['W'], errors='coerce')
        window['GP'] = pd.to_numeric(window['GP'], errors='coerce')
        
        # Applying the new exponential weights
        window['Weighted_Progress'] = window['Playoffs'].map(principled_weight_map).fillna(0)
        
        performance_records.append({
            'Abbr': team_abbr,
            'Win_Pct': window['W'].sum() / window['GP'].sum() if window['GP'].sum() > 0 else 0,
            'Total_Success_Points': window['Weighted_Progress'].sum(),
            'Playoff_Appearances': (window['Weighted_Progress'] > 0).sum(),
            'Cups_Won': (window['Playoffs'] == 'Won Stanley Cup Final').sum()
        })
        
        print(f"✅ Processed: {team_abbr}")
        time.sleep(1.5) 
        
    except Exception as e:
        print(f"❌ Failed {team_abbr}: {e}")

performance_df = pd.DataFrame(performance_records)
performance_df.sort_values('Total_Success_Points', ascending=False).head(10)

✅ Processed: ANA
✅ Processed: ARI
✅ Processed: BOS
✅ Processed: BUF
✅ Processed: CGY
✅ Processed: CAR
✅ Processed: CHI
✅ Processed: COL
✅ Processed: CBJ
✅ Processed: DAL
✅ Processed: DET
✅ Processed: EDM
✅ Processed: FLA
✅ Processed: LAK
✅ Processed: MIN
✅ Processed: MTL
✅ Processed: NSH
✅ Processed: NJD
✅ Processed: NYI
✅ Processed: NYR
✅ Processed: OTT
✅ Processed: PHI
✅ Processed: PIT
✅ Processed: SJS
✅ Processed: STL
✅ Processed: TBL
✅ Processed: TOR
✅ Processed: VAN
✅ Processed: WPG
✅ Processed: WSH


,Abbr,Win_Pct,Total_Success_Points,Playoff_Appearances,Cups_Won
22,PIT,0.564922,116.0,12,3
6,CHI,0.527132,97.0,9,3
10,DET,0.544574,67.0,11,1
0,ANA,0.553295,65.0,11,1
13,LAK,0.496124,63.0,7,2
2,BOS,0.533915,60.0,9,1
23,SJS,0.573643,58.0,12,0
19,NYR,0.538760,54.0,11,0
29,WSH,0.546512,52.0,10,1
25,TBL,0.500000,46.0,7,0


In [21]:
# Create the mapping based on your master_df naming convention
# I've included the standard ones; adjust if your master_df uses slightly different strings
name_mapping = {
    'ANA': 'Anaheim Ducks', 'ARI': 'Arizona Coyotes', 'BOS': 'Boston Bruins',
    'BUF': 'Buffalo Sabres', 'CGY': 'Calgary Flames', 'CAR': 'Carolina Hurricanes',
    'CHI': 'Chicago Blackhawks', 'COL': 'Colorado Avalanche', 'CBJ': 'Columbus Blue Jackets',
    'DAL': 'Dallas Stars', 'DET': 'Detroit Red Wings', 'EDM': 'Edmonton Oilers',
    'FLA': 'Florida Panthers', 'LAK': 'Los Angeles Kings', 'MIN': 'Minnesota Wild',
    'MTL': 'Montreal Canadiens', 'NSH': 'Nashville Predators', 'NJD': 'New Jersey Devils',
    'NYI': 'New York Islanders', 'NYR': 'New York Rangers', 'OTT': 'Ottawa Senators',
    'PHI': 'Philadelphia Flyers', 'PIT': 'Pittsburgh Penguins', 'SJS': 'San Jose Sharks',
    'STL': 'St. Louis Blues', 'TBL': 'Tampa Bay Lightning', 'TOR': 'Toronto Maple Leafs',
    'VAN': 'Vancouver Canucks', 'WPG': 'Winnipeg Jets', 'WSH': 'Washington Capitals'
}

performance_df['Team_Full'] = performance_df['Abbr'].map(name_mapping)

# TEST: Check for any NaNs (meaning an abbreviation was missed)
missing = performance_df[performance_df['Team_Full'].isna()]
if not missing.empty:
    print(f"Missing names for: {missing['Abbr'].tolist()}")
else:
    print("All teams successfully mapped to full names.")

All teams successfully mapped to full names.


In [23]:
# We'll use a 1:1 weighting now because 'Total_Success_Points' 
# already has the 'Exponential' importance of championships baked in.

WIN_PCT_WEIGHT = 100 

performance_df['Success_Index'] = (
    (performance_df['Win_Pct'] * WIN_PCT_WEIGHT) + 
    performance_df['Total_Success_Points']
)

# Sort and View the Final "DNA Validation" Leaderboard
final_rankings = performance_df.sort_values('Success_Index', ascending=False)

print("--- FINAL NHL PERFORMANCE RANKINGS (2005-2017) ---")
display(final_rankings[['Team_Full', 'Win_Pct', 'Total_Success_Points', 'Cups_Won', 'Success_Index']].head(10))

--- FINAL NHL PERFORMANCE RANKINGS (2005-2017) ---


,Team_Full,Win_Pct,Total_Success_Points,Cups_Won,Success_Index
22,Pittsburgh Penguins,0.564922,116.0,3,172.492248
6,Chicago Blackhawks,0.527132,97.0,3,149.713178
10,Detroit Red Wings,0.544574,67.0,1,121.457364
0,Anaheim Ducks,0.553295,65.0,1,120.329457
23,San Jose Sharks,0.573643,58.0,0,115.364341
2,Boston Bruins,0.533915,60.0,1,113.391473
13,Los Angeles Kings,0.496124,63.0,2,112.612403
19,New York Rangers,0.538760,54.0,0,107.875969
29,Washington Capitals,0.546512,52.0,1,106.651163
25,Tampa Bay Lightning,0.500000,46.0,0,96.000000


In [4]:
import pandas as pd
import time
import requests

# --- STEP 1: Give it time (avoid immediate 429) ---
print("⏳ Waiting before request...")
time.sleep(20)

# --- STEP 2: Fetch Washington page ---
url = "https://www.hockey-reference.com/teams/WSH/history.html"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
response.raise_for_status()

# --- STEP 3: Parse table ---
tables = pd.read_html(response.text)
df = tables[0]

# Flatten columns if needed
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(-1)

# Rename columns (same logic as your script)
df = df.rename(columns={
    df.columns[0]: "Season",
    df.columns[3]: "GP",
    df.columns[4]: "W",
    df.columns[13]: "Playoff_Result",
})

# --- STEP 4: Filter to your window ---
df = df[df["Season"].str.contains("-", na=False)].copy()
df["Season_Start"] = df["Season"].str[:4].astype(int)

window = df[
    (df["Season_Start"] >= 2005) & (df["Season_Start"] <= 2024)
].copy()

# --- STEP 5: Add team column ---
window["Abbr"] = "WSH"

# Keep only relevant columns (match your CSV structure)
window = window[["Abbr", "Season", "GP", "W", "Playoff_Result"]]

print(f"✅ Extracted {len(window)} Washington seasons")

# --- STEP 6: Append to existing CSV ---
csv_path = "../data/nhl_performance.csv"

window.to_csv(csv_path, mode="a", header=False, index=False)

print("💾 Washington data appended successfully")

⏳ Waiting before request...


FileNotFoundError: [Errno 2] No such file or directory: 
<!DOCTYPE html>
<html data-version="klecko-" data-root="/home/hr/build" lang="en" class="no-js" >
<head>
    <meta charset="utf-8">
    <meta http-equiv="x-ua-compatible" content="ie=edge">
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=2.0" />
    <link rel="dns-prefetch" href="https://cdn.ssref.net/req/202604204" />
<script>
/* https://docs.osano.com/hc/en-us/articles/22469433444372-Google-Consent-Mode-v2  */
  window.dataLayer = window.dataLayer ||[];
      function gtag(){dataLayer.push(arguments);}
      gtag('consent','default',{
        'ad_storage':'denied',
        'analytics_storage':'denied',
        'ad_user_data':'denied',
        'ad_personalization':'denied',
        'personalization_storage':'denied',
        'functionality_storage':'granted',
        'security_storage':'granted',
        'wait_for_update': 500
      });
      gtag("set", "ads_data_redaction", true);
</scri